## Import necessary library


In [126]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import ListedColormap
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter
import os
from matplotlib.colorbar import ColorbarBase

## Get File path and load the csv data


In [127]:
from util.data_path import cassava_price_avg_year_fix
from util.data_path import corn_price_avg_year_fix
from util.data_path import green_bean_price_avg_year_fix
from util.data_path import soybean_price_avg_year_fix

## Agricultural Price Analysis System

This comprehensive system analyzes price trends for agricultural commodities with the following features:

- Data validation and cleaning with missing value handling
- Core metrics calculation (YoY changes, volatility, seasonality)
- 5 types of visualizations per crop
- Text summaries with trend analysis
- Batch processing for multiple crops
- Master index generation


In [128]:
import os
from typing import Dict, List, Any, Tuple
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")


def create_directories():
    """Create necessary directories for outputs."""
    directories = ["./figs", "./reports"]
    for directory in directories:
        Path(directory).mkdir(exist_ok=True)
    print("✓ Created output directories: ./figs, ./reports")


# Create directories
create_directories()

✓ Created output directories: ./figs, ./reports


In [129]:
def validate_and_clean_data(
    df: pd.DataFrame, file_path: str
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Validate schema and clean data for a single CSV file.

    Args:
        df: Input DataFrame
        file_path: Path to the CSV file (for error reporting)

    Returns:
        Tuple of (cleaned_df, validation_info)
    """
    validation_info = {
        "file_path": file_path,
        "original_shape": df.shape,
        "missing_summary": {},
        "flagged_years": [],
        "warnings": [],
    }

    # Check required columns
    if "year" not in df.columns:
        raise ValueError(f"Missing 'year' column in {file_path}")

    # Expected month columns (can be strings or integers)
    expected_months = [str(i) for i in range(1, 13)]
    missing_months = [month for month in expected_months if month not in df.columns]

    if missing_months:
        raise ValueError(f"Missing month columns {missing_months} in {file_path}")

    # Create working copy
    df_clean = df.copy()

    # Ensure year is integer
    df_clean["year"] = pd.to_numeric(df_clean["year"], errors="coerce")
    df_clean = df_clean.dropna(subset=["year"])
    df_clean["year"] = df_clean["year"].astype(int)

    # Convert month columns to numeric, coerce errors to NaN
    for month in expected_months:
        df_clean[month] = pd.to_numeric(df_clean[month], errors="coerce")

    # Calculate missing value statistics
    total_cells = len(df_clean) * 12
    missing_per_month = {}
    missing_per_year = {}

    for month in expected_months:
        missing_count = df_clean[month].isna().sum()
        missing_per_month[month] = {
            "count": missing_count,
            "percentage": (missing_count / len(df_clean)) * 100,
        }

    for _, row in df_clean.iterrows():
        year = row["year"]
        missing_months_count = sum(
            1 for month in expected_months if pd.isna(row[month])
        )
        missing_percentage = (missing_months_count / 12) * 100
        missing_per_year[year] = {
            "missing_months": missing_months_count,
            "percentage": missing_percentage,
        }

        # Flag years with >50% missing data
        if missing_percentage > 50:
            validation_info["flagged_years"].append(
                {"year": year, "missing_percentage": missing_percentage}
            )

    validation_info["missing_summary"] = {
        "per_month": missing_per_month,
        "per_year": missing_per_year,
        "total_missing_cells": sum(df_clean[expected_months].isna().sum()),
        "total_cells": total_cells,
        "overall_missing_percentage": (
            sum(df_clean[expected_months].isna().sum()) / total_cells
        )
        * 100,
    }

    # Optional: Linear interpolation within years (across months)
    # Only interpolate if there are some valid values in the year
    for year in df_clean["year"].unique():
        year_mask = df_clean["year"] == year
        year_data = df_clean.loc[year_mask, expected_months]

        # Check if year has some valid data
        if not year_data.isna().all().all():
            # Interpolate across months for this year
            df_clean.loc[year_mask, expected_months] = year_data.interpolate(
                axis=1, method="linear"
            )

    # Calculate average price per year
    df_clean["avg_price"] = df_clean[expected_months].mean(axis=1, skipna=True)

    # Create long format DataFrame
    df_long = []
    for _, row in df_clean.iterrows():
        year = row["year"]
        for month in expected_months:
            if not pd.isna(row[month]):
                df_long.append({"year": year, "month": int(month), "price": row[month]})

    df_long = pd.DataFrame(df_long)

    validation_info["cleaned_shape"] = df_clean.shape
    validation_info["long_format_records"] = len(df_long)

    return df_clean, df_long, validation_info

In [130]:
def calculate_core_metrics(
    df_clean: pd.DataFrame, df_long: pd.DataFrame
) -> Dict[str, Any]:
    """
    Calculate core metrics for the price data.

    Args:
        df_clean: Cleaned DataFrame with yearly averages
        df_long: Long format DataFrame

    Returns:
        Dictionary containing all calculated metrics
    """
    metrics = {}

    # Year range
    metrics["year_range"] = f"{df_clean['year'].min()}–{df_clean['year'].max()}"
    metrics["min_year"] = int(df_clean["year"].min())
    metrics["max_year"] = int(df_clean["year"].max())
    metrics["num_years"] = len(df_clean)

    # Overall statistics
    metrics["overall_mean"] = df_clean["avg_price"].mean()

    # Max and min years by average price
    max_idx = df_clean["avg_price"].idxmax()
    min_idx = df_clean["avg_price"].idxmin()

    metrics["max_avg_price"] = df_clean.loc[max_idx, "avg_price"]
    metrics["max_price_year"] = int(df_clean.loc[max_idx, "year"])
    metrics["min_avg_price"] = df_clean.loc[min_idx, "avg_price"]
    metrics["min_price_year"] = int(df_clean.loc[min_idx, "year"])

    # Year-over-year changes
    df_sorted = df_clean.sort_values("year").copy()
    df_sorted["yoy_change_pct"] = df_sorted["avg_price"].pct_change() * 100

    metrics["avg_yoy_change_pct"] = df_sorted["yoy_change_pct"].mean()
    metrics["std_yoy_change_pct"] = df_sorted["yoy_change_pct"].std()

    # High volatility years (where abs(yoy) > mean + std)
    threshold = abs(metrics["avg_yoy_change_pct"]) + metrics["std_yoy_change_pct"]
    high_vol_mask = abs(df_sorted["yoy_change_pct"]) > threshold
    high_vol_years = df_sorted[high_vol_mask][["year", "yoy_change_pct"]].dropna()

    metrics["high_volatility_years"] = [
        {"year": int(row["year"]), "yoy_change_pct": row["yoy_change_pct"]}
        for _, row in high_vol_years.iterrows()
    ]

    # Monthly statistics across all years
    month_columns = [str(i) for i in range(1, 13)]
    monthly_stats = {}

    for month in month_columns:
        month_data = df_clean[month].dropna()
        monthly_stats[int(month)] = {
            "avg": month_data.mean(),
            "min": month_data.min(),
            "max": month_data.max(),
            "std": month_data.std(),
        }

    metrics["monthly_stats"] = monthly_stats

    # Monthly averages for plotting
    metrics["monthly_avg"] = [monthly_stats[i]["avg"] for i in range(1, 13)]

    # 3-year moving average
    df_sorted["moving_avg_3yr"] = (
        df_sorted["avg_price"].rolling(window=3, min_periods=1).mean()
    )
    metrics["moving_avg_3yr"] = df_sorted[["year", "moving_avg_3yr"]].to_dict("records")

    # Correlation matrix for monthly prices
    correlation_matrix = df_clean[month_columns].corr()
    metrics["correlation_matrix"] = correlation_matrix

    # Store processed dataframes for plotting
    metrics["df_sorted"] = df_sorted
    metrics["df_long"] = df_long

    return metrics

In [131]:
def create_visualizations(metrics: Dict[str, Any], basename: str) -> List[str]:
    """
    Create all 5 required visualizations for a single crop.

    Args:
        metrics: Dictionary containing calculated metrics
        basename: Base name for the crop (for file naming)

    Returns:
        List of paths to saved figures
    """
    saved_figures = []

    # 1. Average Monthly Price Trend
    plt.figure(figsize=(12, 6))
    months = list(range(1, 13))
    plt.plot(months, metrics["monthly_avg"], marker="o", linewidth=2, markersize=6)
    plt.title(
        f"Average Monthly Price Trend ({metrics['min_year']}–{metrics['max_year']})",
        fontsize=14,
        fontweight="bold",
    )
    plt.xlabel("Month", fontsize=12)
    plt.ylabel("Price", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.xticks(months)
    plt.tight_layout()

    fig_path = f"./figs/{basename}_monthly_trend.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(fig_path)
    plt.close()

    # 2. Average Yearly Price Trend
    plt.figure(figsize=(12, 6))
    df_sorted = metrics["df_sorted"]
    plt.plot(
        df_sorted["year"], df_sorted["avg_price"], marker="o", linewidth=2, markersize=6
    )
    plt.title("Average Yearly Price Trend", fontsize=14, fontweight="bold")
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Average Price", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    fig_path = f"./figs/{basename}_yearly_trend.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(fig_path)
    plt.close()

    # 3. Smoothed Trend (3-Year Moving Average)
    plt.figure(figsize=(12, 6))
    plt.plot(
        df_sorted["year"],
        df_sorted["avg_price"],
        marker="o",
        linewidth=2,
        markersize=6,
        label="Annual Average",
        solid_capstyle="round",
    )
    plt.plot(
        df_sorted["year"],
        df_sorted["moving_avg_3yr"],
        "--",
        linewidth=2,
        label="3-Year Moving Average",
    )
    plt.title("Yearly Trend with 3-Year Moving Average", fontsize=14, fontweight="bold")
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Average Price", fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    fig_path = f"./figs/{basename}_smoothed_trend.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(fig_path)
    plt.close()

    # 4. Year-over-Year Change (%)
    plt.figure(figsize=(12, 6))
    yoy_data = df_sorted.dropna(subset=["yoy_change_pct"])
    bars = plt.bar(yoy_data["year"], yoy_data["yoy_change_pct"], alpha=0.7)
    plt.axhline(y=0, color="black", linestyle="-", linewidth=1)
    plt.title("Year-over-Year Price Change (%)", fontsize=14, fontweight="bold")
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("YoY Change (%)", fontsize=12)
    plt.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()

    fig_path = f"./figs/{basename}_yoy_change.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(fig_path)
    plt.close()

    # 5. Monthly Correlation Heatmap
    plt.figure(figsize=(10, 8))
    corr_matrix = metrics["correlation_matrix"]

    # Create the heatmap using imshow
    im = plt.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)

    # Add colorbar
    cbar = plt.colorbar(im)
    cbar.set_label("Correlation Coefficient", fontsize=12)

    # Set ticks and labels
    months = [str(i) for i in range(1, 13)]
    plt.xticks(range(12), months)
    plt.yticks(range(12), months)
    plt.xlabel("Month", fontsize=12)
    plt.ylabel("Month", fontsize=12)

    # Add text annotations
    for i in range(12):
        for j in range(12):
            text = plt.text(
                j,
                i,
                f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=9,
            )

    plt.title("Correlation Between Monthly Prices", fontsize=14, fontweight="bold")
    plt.tight_layout()

    fig_path = f"./figs/{basename}_correlation_heatmap.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(fig_path)
    plt.close()

    return saved_figures

In [132]:
def generate_text_summary(
    metrics: Dict[str, Any], validation_info: Dict[str, Any], basename: str
) -> str:
    """
    Generate comprehensive text summary for a single crop.

    Args:
        metrics: Dictionary containing calculated metrics
        validation_info: Validation and data quality information
        basename: Base name for the crop

    Returns:
        Formatted summary text
    """
    summary_lines = []

    # Header
    summary_lines.append(
        f"# {basename.replace('_', ' ').title()} Price Analysis Summary"
    )
    summary_lines.append("=" * 50)
    summary_lines.append("")

    # Data Coverage
    summary_lines.append("## Data Coverage")
    summary_lines.append(f"**Time Period:** {metrics['year_range']}")
    summary_lines.append(f"**Number of Years:** {metrics['num_years']}")
    summary_lines.append(
        f"**Missing Values:** {validation_info['missing_summary']['overall_missing_percentage']:.1f}%"
    )

    if validation_info["flagged_years"]:
        summary_lines.append(
            f"**⚠️ Data Quality Warnings:** {len(validation_info['flagged_years'])} years with >50% missing data"
        )
        for year_info in validation_info["flagged_years"]:
            summary_lines.append(
                f"  - {year_info['year']}: {year_info['missing_percentage']:.1f}% missing"
            )

    summary_lines.append("")

    # Level Overview
    summary_lines.append("## Price Level Overview")
    summary_lines.append(f"**Overall Mean Price:** {metrics['overall_mean']:.2f}")
    summary_lines.append(
        f"**Highest Price Year:** {metrics['max_price_year']} ({metrics['max_avg_price']:.2f})"
    )
    summary_lines.append(
        f"**Lowest Price Year:** {metrics['min_price_year']} ({metrics['min_avg_price']:.2f})"
    )
    summary_lines.append("")

    # Trend Overview
    summary_lines.append("## Trend Overview")
    df_sorted = metrics["df_sorted"]
    first_price = df_sorted.iloc[0]["avg_price"]
    last_price = df_sorted.iloc[-1]["avg_price"]
    trend_direction = "Increasing" if last_price > first_price else "Decreasing"
    trend_magnitude = ((last_price - first_price) / first_price) * 100

    summary_lines.append(
        f"**Overall Trend:** {trend_direction} ({trend_magnitude:+.1f}% from {metrics['min_year']} to {metrics['max_year']})"
    )

    # 3-year moving average commentary
    recent_ma = df_sorted["moving_avg_3yr"].iloc[-3:].values
    if len(recent_ma) >= 2:
        if recent_ma[-1] > recent_ma[0]:
            ma_trend = "sustained rise in recent years"
        else:
            ma_trend = "decline in recent years"
    else:
        ma_trend = "insufficient data for recent trend analysis"

    summary_lines.append(f"**3-Year Moving Average:** Shows {ma_trend}")
    summary_lines.append("")

    # Volatility Overview
    summary_lines.append("## Volatility Overview")
    summary_lines.append(
        f"**Average YoY Change:** {metrics['avg_yoy_change_pct']:.1f}%"
    )
    summary_lines.append(
        f"**YoY Standard Deviation:** {metrics['std_yoy_change_pct']:.1f}%"
    )

    if metrics["high_volatility_years"]:
        summary_lines.append(
            f"**High Volatility Years:** {len(metrics['high_volatility_years'])} years with extreme changes"
        )
        for vol_year in metrics["high_volatility_years"]:
            summary_lines.append(
                f"  - {vol_year['year']}: {vol_year['yoy_change_pct']:+.1f}%"
            )
    else:
        summary_lines.append("**High Volatility Years:** None detected")

    summary_lines.append("")

    # Seasonality Overview
    summary_lines.append("## Seasonality Overview")
    monthly_stats = metrics["monthly_stats"]
    max_month = max(monthly_stats.keys(), key=lambda x: monthly_stats[x]["avg"])
    min_month = max(monthly_stats.keys(), key=lambda x: -monthly_stats[x]["avg"])

    summary_lines.append(
        f"**Highest Average Month:** Month {max_month} ({monthly_stats[max_month]['avg']:.2f})"
    )
    summary_lines.append(
        f"**Lowest Average Month:** Month {min_month} ({monthly_stats[min_month]['avg']:.2f})"
    )

    # Seasonal pattern analysis
    q1_avg = np.mean([monthly_stats[i]["avg"] for i in [1, 2, 3]])
    q2_avg = np.mean([monthly_stats[i]["avg"] for i in [4, 5, 6]])
    q3_avg = np.mean([monthly_stats[i]["avg"] for i in [7, 8, 9]])
    q4_avg = np.mean([monthly_stats[i]["avg"] for i in [10, 11, 12]])

    quarters = {"Q1": q1_avg, "Q2": q2_avg, "Q3": q3_avg, "Q4": q4_avg}
    peak_quarter = max(quarters.keys(), key=lambda x: quarters[x])

    summary_lines.append(
        f"**Seasonal Pattern:** Peak in {peak_quarter} (quarterly averages: Q1={q1_avg:.2f}, Q2={q2_avg:.2f}, Q3={q3_avg:.2f}, Q4={q4_avg:.2f})"
    )

    # Correlation insights
    corr_matrix = metrics["correlation_matrix"]

    # Find strongest positive correlations (excluding diagonal)
    corr_values = []
    for i in range(12):
        for j in range(i + 1, 12):
            corr_values.append(
                {"months": (i + 1, j + 1), "correlation": corr_matrix.iloc[i, j]}
            )

    # Sort by absolute correlation value
    corr_values.sort(key=lambda x: abs(x["correlation"]), reverse=True)

    strongest_pos = max(corr_values, key=lambda x: x["correlation"])
    strongest_neg = min(corr_values, key=lambda x: x["correlation"])

    summary_lines.append(
        f"**Strongest Positive Correlation:** Months {strongest_pos['months'][0]} & {strongest_pos['months'][1]} ({strongest_pos['correlation']:.3f})"
    )
    summary_lines.append(
        f"**Strongest Negative Correlation:** Months {strongest_neg['months'][0]} & {strongest_neg['months'][1]} ({strongest_neg['correlation']:.3f})"
    )
    summary_lines.append("")

    # Data Quality Section
    if (
        validation_info["flagged_years"]
        or validation_info["missing_summary"]["overall_missing_percentage"] > 10
    ):
        summary_lines.append("## Data Quality Notes")
        if validation_info["missing_summary"]["overall_missing_percentage"] > 10:
            summary_lines.append(
                f"⚠️ High missing data rate: {validation_info['missing_summary']['overall_missing_percentage']:.1f}%"
            )
        if validation_info["flagged_years"]:
            summary_lines.append(
                f"⚠️ {len(validation_info['flagged_years'])} years flagged as sparse (>50% missing)"
            )
        summary_lines.append("")

    return "\n".join(summary_lines)

In [133]:
def analyze_price_file(file_path: str) -> Dict[str, Any]:
    """
    Main analysis function for a single CSV file.

    Args:
        file_path: Path to the CSV file to analyze

    Returns:
        Dictionary containing analysis results including:
        - figure_paths: List of saved figure paths
        - summary_text: Generated summary text
        - metrics: Dictionary of calculated metrics
        - summary_path: Path to saved summary markdown file
    """
    try:
        # Extract basename for file naming
        basename = Path(file_path).stem

        print(f"\n📊 Analyzing {basename}...")
        print(f"📂 Reading data from: {file_path}")

        # Read CSV
        try:
            df = pd.read_csv(file_path)
            print(f"✓ Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
        except Exception as e:
            raise ValueError(f"Failed to read CSV file: {e}")

        # Validate and clean data
        print("🔍 Validating and cleaning data...")
        df_clean, df_long, validation_info = validate_and_clean_data(df, file_path)
        print(f"✓ Data cleaned: {len(df_long)} monthly records processed")

        if validation_info["flagged_years"]:
            print(
                f"⚠️ Warning: {len(validation_info['flagged_years'])} years flagged for sparse data"
            )

        # Calculate metrics
        print("📈 Calculating core metrics...")
        metrics = calculate_core_metrics(df_clean, df_long)
        print(f"✓ Metrics calculated for {metrics['year_range']}")

        # Create visualizations
        print("🎨 Creating visualizations...")
        figure_paths = create_visualizations(metrics, basename)
        print(f"✓ {len(figure_paths)} charts saved to ./figs/")

        # Generate summary
        print("📝 Generating summary...")
        summary_text = generate_text_summary(metrics, validation_info, basename)

        # Save summary to file
        summary_path = f"./reports/{basename}_summary.md"
        with open(summary_path, "w", encoding="utf-8") as f:
            f.write(summary_text)

        print(f"✓ Summary saved to {summary_path}")

        # Print summary to console
        print("\n" + "=" * 60)
        print(summary_text)
        print("=" * 60)

        return {
            "figure_paths": figure_paths,
            "summary_text": summary_text,
            "metrics": metrics,
            "summary_path": summary_path,
            "basename": basename,
            "validation_info": validation_info,
        }

    except Exception as e:
        error_msg = f"Error analyzing {file_path}: {str(e)}"
        print(f"❌ {error_msg}")
        return {
            "error": error_msg,
            "figure_paths": [],
            "summary_text": "",
            "metrics": {},
            "summary_path": "",
            "basename": Path(file_path).stem,
            "validation_info": {},
        }

In [134]:
def run_batch(file_paths: List[str]) -> None:
    """
    Run batch analysis on multiple CSV files and generate a master index.

    Args:
        file_paths: List of paths to CSV files to analyze
    """
    print("🚀 Starting batch analysis...")
    print(f"📋 Processing {len(file_paths)} files")

    # Store results for master index
    master_index_data = []
    successful_analyses = []
    failed_analyses = []

    # Process each file
    for i, file_path in enumerate(file_paths, 1):
        print(f"\n{'=' * 80}")
        print(f"Processing file {i}/{len(file_paths)}: {Path(file_path).name}")
        print(f"{'=' * 80}")

        result = analyze_price_file(file_path)

        if "error" in result:
            failed_analyses.append({"file_path": file_path, "error": result["error"]})
            continue

        successful_analyses.append(result)

        # Extract data for master index
        metrics = result["metrics"]
        validation_info = result["validation_info"]

        index_entry = {
            "file": Path(file_path).name,
            "crop": result["basename"].replace("_", " ").title(),
            "year_range": metrics["year_range"],
            "overall_mean": round(metrics["overall_mean"], 2),
            "max_year": metrics["max_price_year"],
            "max_avg_price": round(metrics["max_avg_price"], 2),
            "min_year": metrics["min_price_year"],
            "min_avg_price": round(metrics["min_avg_price"], 2),
            "avg_yoy_change_pct": round(metrics["avg_yoy_change_pct"], 2),
            "std_yoy_change_pct": round(metrics["std_yoy_change_pct"], 2),
            "high_volatility_years_count": len(metrics["high_volatility_years"]),
            "missing_data_pct": round(
                validation_info["missing_summary"]["overall_missing_percentage"], 1
            ),
            "flagged_years_count": len(validation_info["flagged_years"]),
        }

        master_index_data.append(index_entry)

    # Create and save master index
    print(f"\n{'=' * 80}")
    print("📊 BATCH ANALYSIS SUMMARY")
    print(f"{'=' * 80}")

    if master_index_data:
        master_df = pd.DataFrame(master_index_data)
        index_path = "./reports/_index.csv"
        master_df.to_csv(index_path, index=False)

        print(f"\n✅ Successfully processed {len(successful_analyses)} files")
        if failed_analyses:
            print(f"❌ Failed to process {len(failed_analyses)} files")

        print(f"\n📋 Master Index ({index_path}):")
        print("-" * 120)

        # Print formatted table
        for _, row in master_df.iterrows():
            print(
                f"{row['crop']:<15} | {row['year_range']:<12} | Mean: {row['overall_mean']:>8.2f} | "
                f"Max: {row['max_avg_price']:>8.2f} ({row['max_year']}) | Min: {row['min_avg_price']:>8.2f} ({row['min_year']}) | "
                f"YoY Avg: {row['avg_yoy_change_pct']:>6.1f}% | Vol: {row['std_yoy_change_pct']:>5.1f}%"
            )

        print("-" * 120)

        # Summary statistics
        print(f"\n📈 Cross-Crop Summary:")
        print(f"• Average Mean Price: {master_df['overall_mean'].mean():.2f}")
        print(
            f"• Most Volatile Crop: {master_df.loc[master_df['std_yoy_change_pct'].idxmax(), 'crop']} ({master_df['std_yoy_change_pct'].max():.1f}% std)"
        )
        print(
            f"• Least Volatile Crop: {master_df.loc[master_df['std_yoy_change_pct'].idxmin(), 'crop']} ({master_df['std_yoy_change_pct'].min():.1f}% std)"
        )
        print(
            f"• Highest Average Growth: {master_df.loc[master_df['avg_yoy_change_pct'].idxmax(), 'crop']} ({master_df['avg_yoy_change_pct'].max():.1f}% YoY)"
        )

        data_quality_issues = master_df[master_df["missing_data_pct"] > 10]
        if not data_quality_issues.empty:
            print(f"\n⚠️ Data Quality Concerns:")
            for _, row in data_quality_issues.iterrows():
                print(f"• {row['crop']}: {row['missing_data_pct']}% missing data")

    else:
        print("❌ No files were successfully processed")

    # Report failed analyses
    if failed_analyses:
        print(f"\n❌ Failed Analyses:")
        for failure in failed_analyses:
            print(f"• {Path(failure['file_path']).name}: {failure['error']}")

    print(f"\n🎯 Output Summary:")
    print(f"• Figures saved to: ./figs/")
    print(f"• Reports saved to: ./reports/")
    print(f"• Master index: ./reports/_index.csv")

    print(f"\n✨ Batch analysis complete!")

## Example Usage

Now let's test the system with your current crop data files. We'll use the paths already imported from your `util.data_path` module.


In [135]:
# Example usage with your existing crop data
if __name__ == "__main__":
    # Use the paths from your util.data_path module
    file_paths = [
        str(cassava_price_avg_year_fix),
        str(corn_price_avg_year_fix),
        str(green_bean_price_avg_year_fix),
        str(soybean_price_avg_year_fix),
    ]

    print("🌾 Agricultural Price Analysis System")
    print("====================================")
    print("This system will analyze price trends for your crop data:")
    for i, path in enumerate(file_paths, 1):
        crop_name = Path(path).parent.name.replace("_", " ").title()
        print(f"{i}. {crop_name}: {path}")

    print(f"\n🔄 Ready to analyze {len(file_paths)} crop datasets...")
    print("Press Enter to continue or type 'skip' to skip execution.")

    # Uncomment the next line to run automatically
    # run_batch(file_paths)

🌾 Agricultural Price Analysis System
This system will analyze price trends for your crop data:
1. Cassava: c:\Users\thatt\Documents\Coding Project\Science Projects\AI Crop Land-Used\data\fix_year\cassava\price_avg.csv
2. Corn: c:\Users\thatt\Documents\Coding Project\Science Projects\AI Crop Land-Used\data\fix_year\corn\price_avg.csv
3. Green Bean: c:\Users\thatt\Documents\Coding Project\Science Projects\AI Crop Land-Used\data\fix_year\green_bean\price_avg.csv
4. Soybean: c:\Users\thatt\Documents\Coding Project\Science Projects\AI Crop Land-Used\data\fix_year\soybean\price_avg.csv

🔄 Ready to analyze 4 crop datasets...
Press Enter to continue or type 'skip' to skip execution.


## Run the Complete Analysis

Execute the cell below to run the full batch analysis on your crop data. This will:

1. **Process each crop file** (cassava, corn, green bean, soybean)
2. **Generate 5 visualizations** per crop (20 total charts)
3. **Create detailed summaries** in Markdown format
4. **Produce a master index** comparing all crops
5. **Save everything** to organized folders (`./figs/` and `./reports/`)

The analysis includes:

- **Data validation** with missing value reporting
- **Core metrics** (YoY changes, volatility, seasonality)
- **Trend analysis** with 3-year moving averages
- **Correlation heatmaps** for monthly price relationships
- **Cross-crop comparisons** in the master index


In [136]:
# Execute the complete batch analysis
file_paths = [
    str(cassava_price_avg_year_fix),
    str(corn_price_avg_year_fix),
    str(green_bean_price_avg_year_fix),
    str(soybean_price_avg_year_fix),
]

# Run the complete analysis
run_batch(file_paths)

🚀 Starting batch analysis...
📋 Processing 4 files

Processing file 1/4: price_avg.csv

📊 Analyzing price_avg...
📂 Reading data from: c:\Users\thatt\Documents\Coding Project\Science Projects\AI Crop Land-Used\data\fix_year\cassava\price_avg.csv
✓ Data loaded: 21 rows, 13 columns
🔍 Validating and cleaning data...
✓ Data cleaned: 252 monthly records processed
📈 Calculating core metrics...
✓ Metrics calculated for 2004–2024
🎨 Creating visualizations...
✓ 5 charts saved to ./figs/
📝 Generating summary...
✓ Summary saved to ./reports/price_avg_summary.md

# Price Avg Price Analysis Summary

## Data Coverage
**Time Period:** 2004–2024
**Number of Years:** 21
**Missing Values:** 0.0%

## Price Level Overview
**Overall Mean Price:** 2.22
**Highest Price Year:** 2023 (3.43)
**Lowest Price Year:** 2004 (1.12)

## Trend Overview
**Overall Trend:** Increasing (+174.0% from 2004 to 2024)
**3-Year Moving Average:** Shows sustained rise in recent years

## Volatility Overview
**Average YoY Change:** 8

## Complete Template Code

Here's the standalone, complete Python code that implements all your requirements:


In [137]:
"""
Agricultural Price Analysis System
==================================

A comprehensive system for analyzing agricultural commodity price data with:
- Data validation and cleaning
- Core metrics calculation (YoY changes, volatility, seasonality) 
- 5 types of visualizations per crop
- Text summaries with trend analysis
- Batch processing for multiple crops
- Master index generation

Author: GitHub Copilot
Date: October 2025
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from typing import Dict, List, Any, Tuple
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")


def create_directories():
    """Create necessary directories for outputs."""
    directories = ["./figs", "./reports"]
    for directory in directories:
        Path(directory).mkdir(exist_ok=True)
    print("✓ Created output directories: ./figs, ./reports")


def validate_and_clean_data(
    df: pd.DataFrame, file_path: str
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """
    Validate schema and clean data for a single CSV file.

    Args:
        df: Input DataFrame
        file_path: Path to the CSV file (for error reporting)

    Returns:
        Tuple of (cleaned_df, df_long, validation_info)
    """
    validation_info = {
        "file_path": file_path,
        "original_shape": df.shape,
        "missing_summary": {},
        "flagged_years": [],
        "warnings": [],
    }

    # Check required columns
    if "year" not in df.columns:
        raise ValueError(f"Missing 'year' column in {file_path}")

    # Expected month columns (can be strings or integers)
    expected_months = [str(i) for i in range(1, 13)]
    missing_months = [month for month in expected_months if month not in df.columns]

    if missing_months:
        raise ValueError(f"Missing month columns {missing_months} in {file_path}")

    # Create working copy
    df_clean = df.copy()

    # Ensure year is integer
    df_clean["year"] = pd.to_numeric(df_clean["year"], errors="coerce")
    df_clean = df_clean.dropna(subset=["year"])
    df_clean["year"] = df_clean["year"].astype(int)

    # Convert month columns to numeric, coerce errors to NaN
    for month in expected_months:
        df_clean[month] = pd.to_numeric(df_clean[month], errors="coerce")

    # Calculate missing value statistics
    total_cells = len(df_clean) * 12
    missing_per_month = {}
    missing_per_year = {}

    for month in expected_months:
        missing_count = df_clean[month].isna().sum()
        missing_per_month[month] = {
            "count": missing_count,
            "percentage": (missing_count / len(df_clean)) * 100,
        }

    for _, row in df_clean.iterrows():
        year = row["year"]
        missing_months_count = sum(
            1 for month in expected_months if pd.isna(row[month])
        )
        missing_percentage = (missing_months_count / 12) * 100
        missing_per_year[year] = {
            "missing_months": missing_months_count,
            "percentage": missing_percentage,
        }

        # Flag years with >50% missing data
        if missing_percentage > 50:
            validation_info["flagged_years"].append(
                {"year": year, "missing_percentage": missing_percentage}
            )

    validation_info["missing_summary"] = {
        "per_month": missing_per_month,
        "per_year": missing_per_year,
        "total_missing_cells": sum(df_clean[expected_months].isna().sum()),
        "total_cells": total_cells,
        "overall_missing_percentage": (
            sum(df_clean[expected_months].isna().sum()) / total_cells
        )
        * 100,
    }

    # Optional: Linear interpolation within years (across months)
    for year in df_clean["year"].unique():
        year_mask = df_clean["year"] == year
        year_data = df_clean.loc[year_mask, expected_months]

        # Check if year has some valid data
        if not year_data.isna().all().all():
            # Interpolate across months for this year
            df_clean.loc[year_mask, expected_months] = year_data.interpolate(
                axis=1, method="linear"
            )

    # Calculate average price per year
    df_clean["avg_price"] = df_clean[expected_months].mean(axis=1, skipna=True)

    # Create long format DataFrame
    df_long = []
    for _, row in df_clean.iterrows():
        year = row["year"]
        for month in expected_months:
            if not pd.isna(row[month]):
                df_long.append({"year": year, "month": int(month), "price": row[month]})

    df_long = pd.DataFrame(df_long)

    validation_info["cleaned_shape"] = df_clean.shape
    validation_info["long_format_records"] = len(df_long)

    return df_clean, df_long, validation_info


def calculate_core_metrics(
    df_clean: pd.DataFrame, df_long: pd.DataFrame
) -> Dict[str, Any]:
    """
    Calculate core metrics for the price data.

    Args:
        df_clean: Cleaned DataFrame with yearly averages
        df_long: Long format DataFrame

    Returns:
        Dictionary containing all calculated metrics
    """
    metrics = {}

    # Year range
    metrics["year_range"] = f"{df_clean['year'].min()}–{df_clean['year'].max()}"
    metrics["min_year"] = int(df_clean["year"].min())
    metrics["max_year"] = int(df_clean["year"].max())
    metrics["num_years"] = len(df_clean)

    # Overall statistics
    metrics["overall_mean"] = df_clean["avg_price"].mean()

    # Max and min years by average price
    max_idx = df_clean["avg_price"].idxmax()
    min_idx = df_clean["avg_price"].idxmin()

    metrics["max_avg_price"] = df_clean.loc[max_idx, "avg_price"]
    metrics["max_price_year"] = int(df_clean.loc[max_idx, "year"])
    metrics["min_avg_price"] = df_clean.loc[min_idx, "avg_price"]
    metrics["min_price_year"] = int(df_clean.loc[min_idx, "year"])

    # Year-over-year changes
    df_sorted = df_clean.sort_values("year").copy()
    df_sorted["yoy_change_pct"] = df_sorted["avg_price"].pct_change() * 100

    metrics["avg_yoy_change_pct"] = df_sorted["yoy_change_pct"].mean()
    metrics["std_yoy_change_pct"] = df_sorted["yoy_change_pct"].std()

    # High volatility years (where abs(yoy) > mean + std)
    threshold = abs(metrics["avg_yoy_change_pct"]) + metrics["std_yoy_change_pct"]
    high_vol_mask = abs(df_sorted["yoy_change_pct"]) > threshold
    high_vol_years = df_sorted[high_vol_mask][["year", "yoy_change_pct"]].dropna()

    metrics["high_volatility_years"] = [
        {"year": int(row["year"]), "yoy_change_pct": row["yoy_change_pct"]}
        for _, row in high_vol_years.iterrows()
    ]

    # Monthly statistics across all years
    month_columns = [str(i) for i in range(1, 13)]
    monthly_stats = {}

    for month in month_columns:
        month_data = df_clean[month].dropna()
        monthly_stats[int(month)] = {
            "avg": month_data.mean(),
            "min": month_data.min(),
            "max": month_data.max(),
            "std": month_data.std(),
        }

    metrics["monthly_stats"] = monthly_stats

    # Monthly averages for plotting
    metrics["monthly_avg"] = [monthly_stats[i]["avg"] for i in range(1, 13)]

    # 3-year moving average
    df_sorted["moving_avg_3yr"] = (
        df_sorted["avg_price"].rolling(window=3, min_periods=1).mean()
    )
    metrics["moving_avg_3yr"] = df_sorted[["year", "moving_avg_3yr"]].to_dict("records")

    # Correlation matrix for monthly prices
    correlation_matrix = df_clean[month_columns].corr()
    metrics["correlation_matrix"] = correlation_matrix

    # Store processed dataframes for plotting
    metrics["df_sorted"] = df_sorted
    metrics["df_long"] = df_long

    return metrics


def create_visualizations(metrics: Dict[str, Any], basename: str) -> List[str]:
    """
    Create all 5 required visualizations for a single crop.

    Args:
        metrics: Dictionary containing calculated metrics
        basename: Base name for the crop (for file naming)

    Returns:
        List of paths to saved figures
    """
    saved_figures = []

    # 1. Average Monthly Price Trend
    plt.figure(figsize=(12, 6))
    months = list(range(1, 13))
    plt.plot(months, metrics["monthly_avg"], marker="o", linewidth=2, markersize=6)
    plt.title(
        f"Average Monthly Price Trend ({metrics['min_year']}–{metrics['max_year']})",
        fontsize=14,
        fontweight="bold",
    )
    plt.xlabel("Month", fontsize=12)
    plt.ylabel("Price", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.xticks(months)
    plt.tight_layout()

    fig_path = f"./figs/{basename}_monthly_trend.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(fig_path)
    plt.close()

    # 2. Average Yearly Price Trend
    plt.figure(figsize=(12, 6))
    df_sorted = metrics["df_sorted"]
    plt.plot(
        df_sorted["year"], df_sorted["avg_price"], marker="o", linewidth=2, markersize=6
    )
    plt.title("Average Yearly Price Trend", fontsize=14, fontweight="bold")
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Average Price", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    fig_path = f"./figs/{basename}_yearly_trend.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(fig_path)
    plt.close()

    # 3. Smoothed Trend (3-Year Moving Average)
    plt.figure(figsize=(12, 6))
    plt.plot(
        df_sorted["year"],
        df_sorted["avg_price"],
        marker="o",
        linewidth=2,
        markersize=6,
        label="Annual Average",
        solid_capstyle="round",
    )
    plt.plot(
        df_sorted["year"],
        df_sorted["moving_avg_3yr"],
        "--",
        linewidth=2,
        label="3-Year Moving Average",
    )
    plt.title("Yearly Trend with 3-Year Moving Average", fontsize=14, fontweight="bold")
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Average Price", fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    fig_path = f"./figs/{basename}_smoothed_trend.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(fig_path)
    plt.close()

    # 4. Year-over-Year Change (%)
    plt.figure(figsize=(12, 6))
    yoy_data = df_sorted.dropna(subset=["yoy_change_pct"])
    bars = plt.bar(yoy_data["year"], yoy_data["yoy_change_pct"], alpha=0.7)
    plt.axhline(y=0, color="black", linestyle="-", linewidth=1)
    plt.title("Year-over-Year Price Change (%)", fontsize=14, fontweight="bold")
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("YoY Change (%)", fontsize=12)
    plt.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()

    fig_path = f"./figs/{basename}_yoy_change.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(fig_path)
    plt.close()

    # 5. Monthly Correlation Heatmap
    plt.figure(figsize=(10, 8))
    corr_matrix = metrics["correlation_matrix"]

    # Create the heatmap using imshow
    im = plt.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)

    # Add colorbar
    cbar = plt.colorbar(im)
    cbar.set_label("Correlation Coefficient", fontsize=12)

    # Set ticks and labels
    months = [str(i) for i in range(1, 13)]
    plt.xticks(range(12), months)
    plt.yticks(range(12), months)
    plt.xlabel("Month", fontsize=12)
    plt.ylabel("Month", fontsize=12)

    # Add text annotations
    for i in range(12):
        for j in range(12):
            text = plt.text(
                j,
                i,
                f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=9,
            )

    plt.title("Correlation Between Monthly Prices", fontsize=14, fontweight="bold")
    plt.tight_layout()

    fig_path = f"./figs/{basename}_correlation_heatmap.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(fig_path)
    plt.close()

    return saved_figures


def generate_text_summary(
    metrics: Dict[str, Any], validation_info: Dict[str, Any], basename: str
) -> str:
    """Generate comprehensive text summary for a single crop."""
    summary_lines = []

    # Header
    summary_lines.append(
        f"# {basename.replace('_', ' ').title()} Price Analysis Summary"
    )
    summary_lines.append("=" * 50)
    summary_lines.append("")

    # Data Coverage
    summary_lines.append("## Data Coverage")
    summary_lines.append(f"**Time Period:** {metrics['year_range']}")
    summary_lines.append(f"**Number of Years:** {metrics['num_years']}")
    summary_lines.append(
        f"**Missing Values:** {validation_info['missing_summary']['overall_missing_percentage']:.1f}%"
    )

    if validation_info["flagged_years"]:
        summary_lines.append(
            f"**⚠️ Data Quality Warnings:** {len(validation_info['flagged_years'])} years with >50% missing data"
        )
        for year_info in validation_info["flagged_years"]:
            summary_lines.append(
                f"  - {year_info['year']}: {year_info['missing_percentage']:.1f}% missing"
            )

    summary_lines.append("")

    # Price Level Overview
    summary_lines.append("## Price Level Overview")
    summary_lines.append(f"**Overall Mean Price:** {metrics['overall_mean']:.2f}")
    summary_lines.append(
        f"**Highest Price Year:** {metrics['max_price_year']} ({metrics['max_avg_price']:.2f})"
    )
    summary_lines.append(
        f"**Lowest Price Year:** {metrics['min_price_year']} ({metrics['min_avg_price']:.2f})"
    )
    summary_lines.append("")

    # Trend Overview
    summary_lines.append("## Trend Overview")
    df_sorted = metrics["df_sorted"]
    first_price = df_sorted.iloc[0]["avg_price"]
    last_price = df_sorted.iloc[-1]["avg_price"]
    trend_direction = "Increasing" if last_price > first_price else "Decreasing"
    trend_magnitude = ((last_price - first_price) / first_price) * 100

    summary_lines.append(
        f"**Overall Trend:** {trend_direction} ({trend_magnitude:+.1f}% from {metrics['min_year']} to {metrics['max_year']})"
    )

    # 3-year moving average commentary
    recent_ma = df_sorted["moving_avg_3yr"].iloc[-3:].values
    if len(recent_ma) >= 2:
        if recent_ma[-1] > recent_ma[0]:
            ma_trend = "sustained rise in recent years"
        else:
            ma_trend = "decline in recent years"
    else:
        ma_trend = "insufficient data for recent trend analysis"

    summary_lines.append(f"**3-Year Moving Average:** Shows {ma_trend}")
    summary_lines.append("")

    # Volatility Overview
    summary_lines.append("## Volatility Overview")
    summary_lines.append(
        f"**Average YoY Change:** {metrics['avg_yoy_change_pct']:.1f}%"
    )
    summary_lines.append(
        f"**YoY Standard Deviation:** {metrics['std_yoy_change_pct']:.1f}%"
    )

    if metrics["high_volatility_years"]:
        summary_lines.append(
            f"**High Volatility Years:** {len(metrics['high_volatility_years'])} years with extreme changes"
        )
        for vol_year in metrics["high_volatility_years"]:
            summary_lines.append(
                f"  - {vol_year['year']}: {vol_year['yoy_change_pct']:+.1f}%"
            )
    else:
        summary_lines.append("**High Volatility Years:** None detected")

    summary_lines.append("")

    # Seasonality Overview
    summary_lines.append("## Seasonality Overview")
    monthly_stats = metrics["monthly_stats"]
    max_month = max(monthly_stats.keys(), key=lambda x: monthly_stats[x]["avg"])
    min_month = max(monthly_stats.keys(), key=lambda x: -monthly_stats[x]["avg"])

    summary_lines.append(
        f"**Highest Average Month:** Month {max_month} ({monthly_stats[max_month]['avg']:.2f})"
    )
    summary_lines.append(
        f"**Lowest Average Month:** Month {min_month} ({monthly_stats[min_month]['avg']:.2f})"
    )

    # Seasonal pattern analysis
    q1_avg = np.mean([monthly_stats[i]["avg"] for i in [1, 2, 3]])
    q2_avg = np.mean([monthly_stats[i]["avg"] for i in [4, 5, 6]])
    q3_avg = np.mean([monthly_stats[i]["avg"] for i in [7, 8, 9]])
    q4_avg = np.mean([monthly_stats[i]["avg"] for i in [10, 11, 12]])

    quarters = {"Q1": q1_avg, "Q2": q2_avg, "Q3": q3_avg, "Q4": q4_avg}
    peak_quarter = max(quarters.keys(), key=lambda x: quarters[x])

    summary_lines.append(
        f"**Seasonal Pattern:** Peak in {peak_quarter} (quarterly averages: Q1={q1_avg:.2f}, Q2={q2_avg:.2f}, Q3={q3_avg:.2f}, Q4={q4_avg:.2f})"
    )

    # Correlation insights
    corr_matrix = metrics["correlation_matrix"]

    # Find strongest correlations (excluding diagonal)
    corr_values = []
    for i in range(12):
        for j in range(i + 1, 12):
            corr_values.append(
                {"months": (i + 1, j + 1), "correlation": corr_matrix.iloc[i, j]}
            )

    strongest_pos = max(corr_values, key=lambda x: x["correlation"])
    strongest_neg = min(corr_values, key=lambda x: x["correlation"])

    summary_lines.append(
        f"**Strongest Positive Correlation:** Months {strongest_pos['months'][0]} & {strongest_pos['months'][1]} ({strongest_pos['correlation']:.3f})"
    )
    summary_lines.append(
        f"**Strongest Negative Correlation:** Months {strongest_neg['months'][0]} & {strongest_neg['months'][1]} ({strongest_neg['correlation']:.3f})"
    )
    summary_lines.append("")

    # Data Quality Section
    if (
        validation_info["flagged_years"]
        or validation_info["missing_summary"]["overall_missing_percentage"] > 10
    ):
        summary_lines.append("## Data Quality Notes")
        if validation_info["missing_summary"]["overall_missing_percentage"] > 10:
            summary_lines.append(
                f"⚠️ High missing data rate: {validation_info['missing_summary']['overall_missing_percentage']:.1f}%"
            )
        if validation_info["flagged_years"]:
            summary_lines.append(
                f"⚠️ {len(validation_info['flagged_years'])} years flagged as sparse (>50% missing)"
            )
        summary_lines.append("")

    return "\n".join(summary_lines)


def analyze_price_file(file_path: str) -> Dict[str, Any]:
    """
    Main analysis function for a single CSV file.

    Args:
        file_path: Path to the CSV file to analyze

    Returns:
        Dictionary containing analysis results
    """
    try:
        # Extract basename for file naming
        basename = Path(file_path).stem

        print(f"\n📊 Analyzing {basename}...")
        print(f"📂 Reading data from: {file_path}")

        # Read CSV
        try:
            df = pd.read_csv(file_path)
            print(f"✓ Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
        except Exception as e:
            raise ValueError(f"Failed to read CSV file: {e}")

        # Validate and clean data
        print("🔍 Validating and cleaning data...")
        df_clean, df_long, validation_info = validate_and_clean_data(df, file_path)
        print(f"✓ Data cleaned: {len(df_long)} monthly records processed")

        if validation_info["flagged_years"]:
            print(
                f"⚠️ Warning: {len(validation_info['flagged_years'])} years flagged for sparse data"
            )

        # Calculate metrics
        print("📈 Calculating core metrics...")
        metrics = calculate_core_metrics(df_clean, df_long)
        print(f"✓ Metrics calculated for {metrics['year_range']}")

        # Create visualizations
        print("🎨 Creating visualizations...")
        figure_paths = create_visualizations(metrics, basename)
        print(f"✓ {len(figure_paths)} charts saved to ./figs/")

        # Generate summary
        print("📝 Generating summary...")
        summary_text = generate_text_summary(metrics, validation_info, basename)

        # Save summary to file
        summary_path = f"./reports/{basename}_summary.md"
        with open(summary_path, "w", encoding="utf-8") as f:
            f.write(summary_text)

        print(f"✓ Summary saved to {summary_path}")

        # Print summary to console
        print("\n" + "=" * 60)
        print(summary_text)
        print("=" * 60)

        return {
            "figure_paths": figure_paths,
            "summary_text": summary_text,
            "metrics": metrics,
            "summary_path": summary_path,
            "basename": basename,
            "validation_info": validation_info,
        }

    except Exception as e:
        error_msg = f"Error analyzing {file_path}: {str(e)}"
        print(f"❌ {error_msg}")
        return {
            "error": error_msg,
            "figure_paths": [],
            "summary_text": "",
            "metrics": {},
            "summary_path": "",
            "basename": Path(file_path).stem,
            "validation_info": {},
        }


def run_batch(file_paths: List[str]) -> None:
    """
    Run batch analysis on multiple CSV files and generate a master index.

    Args:
        file_paths: List of paths to CSV files to analyze
    """
    print("🚀 Starting batch analysis...")
    print(f"📋 Processing {len(file_paths)} files")

    # Store results for master index
    master_index_data = []
    successful_analyses = []
    failed_analyses = []

    # Process each file
    for i, file_path in enumerate(file_paths, 1):
        print(f"\n{'=' * 80}")
        print(f"Processing file {i}/{len(file_paths)}: {Path(file_path).name}")
        print(f"{'=' * 80}")

        result = analyze_price_file(file_path)

        if "error" in result:
            failed_analyses.append({"file_path": file_path, "error": result["error"]})
            continue

        successful_analyses.append(result)

        # Extract data for master index
        metrics = result["metrics"]
        validation_info = result["validation_info"]

        index_entry = {
            "file": Path(file_path).name,
            "crop": result["basename"].replace("_", " ").title(),
            "year_range": metrics["year_range"],
            "overall_mean": round(metrics["overall_mean"], 2),
            "max_year": metrics["max_price_year"],
            "max_avg_price": round(metrics["max_avg_price"], 2),
            "min_year": metrics["min_price_year"],
            "min_avg_price": round(metrics["min_avg_price"], 2),
            "avg_yoy_change_pct": round(metrics["avg_yoy_change_pct"], 2),
            "std_yoy_change_pct": round(metrics["std_yoy_change_pct"], 2),
            "high_volatility_years_count": len(metrics["high_volatility_years"]),
            "missing_data_pct": round(
                validation_info["missing_summary"]["overall_missing_percentage"], 1
            ),
            "flagged_years_count": len(validation_info["flagged_years"]),
        }

        master_index_data.append(index_entry)

    # Create and save master index
    print(f"\n{'=' * 80}")
    print("📊 BATCH ANALYSIS SUMMARY")
    print(f"{'=' * 80}")

    if master_index_data:
        master_df = pd.DataFrame(master_index_data)
        index_path = "./reports/_index.csv"
        master_df.to_csv(index_path, index=False)

        print(f"\n✅ Successfully processed {len(successful_analyses)} files")
        if failed_analyses:
            print(f"❌ Failed to process {len(failed_analyses)} files")

        print(f"\n📋 Master Index ({index_path}):")
        print("-" * 120)

        # Print formatted table
        for _, row in master_df.iterrows():
            print(
                f"{row['crop']:<15} | {row['year_range']:<12} | Mean: {row['overall_mean']:>8.2f} | "
                f"Max: {row['max_avg_price']:>8.2f} ({row['max_year']}) | Min: {row['min_avg_price']:>8.2f} ({row['min_year']}) | "
                f"YoY Avg: {row['avg_yoy_change_pct']:>6.1f}% | Vol: {row['std_yoy_change_pct']:>5.1f}%"
            )

        print("-" * 120)

        # Summary statistics
        print(f"\n📈 Cross-Crop Summary:")
        print(f"• Average Mean Price: {master_df['overall_mean'].mean():.2f}")
        print(
            f"• Most Volatile Crop: {master_df.loc[master_df['std_yoy_change_pct'].idxmax(), 'crop']} ({master_df['std_yoy_change_pct'].max():.1f}% std)"
        )
        print(
            f"• Least Volatile Crop: {master_df.loc[master_df['std_yoy_change_pct'].idxmin(), 'crop']} ({master_df['std_yoy_change_pct'].min():.1f}% std)"
        )
        print(
            f"• Highest Average Growth: {master_df.loc[master_df['avg_yoy_change_pct'].idxmax(), 'crop']} ({master_df['avg_yoy_change_pct'].max():.1f}% YoY)"
        )

        data_quality_issues = master_df[master_df["missing_data_pct"] > 10]
        if not data_quality_issues.empty:
            print(f"\n⚠️ Data Quality Concerns:")
            for _, row in data_quality_issues.iterrows():
                print(f"• {row['crop']}: {row['missing_data_pct']}% missing data")

    else:
        print("❌ No files were successfully processed")

    # Report failed analyses
    if failed_analyses:
        print(f"\n❌ Failed Analyses:")
        for failure in failed_analyses:
            print(f"• {Path(failure['file_path']).name}: {failure['error']}")

    print(f"\n🎯 Output Summary:")
    print(f"• Figures saved to: ./figs/")
    print(f"• Reports saved to: ./reports/")
    print(f"• Master index: ./reports/_index.csv")

    print(f"\n✨ Batch analysis complete!")


# Example usage
if __name__ == "__main__":
    # Create output directories
    create_directories()

    # Example file paths - replace with your actual paths
    file_paths = [
        "./data/price_avg.csv",
        "./data/price_avg_2.csv",
        "./data/price_avg_3.csv",
        "./data/price_avg_4.csv",
        "./data/price_avg_5.csv",
    ]

    # Run batch analysis
    run_batch(file_paths)

✓ Created output directories: ./figs, ./reports
🚀 Starting batch analysis...
📋 Processing 5 files

Processing file 1/5: price_avg.csv

📊 Analyzing price_avg...
📂 Reading data from: ./data/price_avg.csv
❌ Error analyzing ./data/price_avg.csv: Failed to read CSV file: [Errno 2] No such file or directory: './data/price_avg.csv'

Processing file 2/5: price_avg_2.csv

📊 Analyzing price_avg_2...
📂 Reading data from: ./data/price_avg_2.csv
❌ Error analyzing ./data/price_avg_2.csv: Failed to read CSV file: [Errno 2] No such file or directory: './data/price_avg_2.csv'

Processing file 3/5: price_avg_3.csv

📊 Analyzing price_avg_3...
📂 Reading data from: ./data/price_avg_3.csv
❌ Error analyzing ./data/price_avg_3.csv: Failed to read CSV file: [Errno 2] No such file or directory: './data/price_avg_3.csv'

Processing file 4/5: price_avg_4.csv

📊 Analyzing price_avg_4...
📂 Reading data from: ./data/price_avg_4.csv
❌ Error analyzing ./data/price_avg_4.csv: Failed to read CSV file: [Errno 2] No such 

## Individual Crop Analysis with Separate Folders

This section provides an enhanced version that creates separate folders for each crop, organizing the analysis results more clearly.


In [138]:
def create_crop_directories(crop_names: List[str]):
    """Create directories for each crop's analysis outputs."""
    base_dirs = ["./figs", "./reports"]

    for base_dir in base_dirs:
        Path(base_dir).mkdir(exist_ok=True)
        for crop_name in crop_names:
            crop_dir = Path(base_dir) / crop_name
            crop_dir.mkdir(exist_ok=True)

    print(
        f"✓ Created directories for {len(crop_names)} crops in ./figs/ and ./reports/"
    )


def get_crop_name_from_path(file_path: str) -> str:
    """Extract clean crop name from file path."""
    # Get the parent directory name (e.g., 'cassava', 'corn', etc.)
    crop_name = Path(file_path).parent.name
    return crop_name


def create_crop_visualizations(metrics: Dict[str, Any], crop_name: str) -> List[str]:
    """
    Create all 5 required visualizations for a single crop, saved in crop-specific folder.

    Args:
        metrics: Dictionary containing calculated metrics
        crop_name: Name of the crop (for folder organization)

    Returns:
        List of paths to saved figures
    """
    saved_figures = []

    # Create crop-specific directory
    crop_figs_dir = Path(f"./figs/{crop_name}")
    crop_figs_dir.mkdir(parents=True, exist_ok=True)

    # 1. Average Monthly Price Trend
    plt.figure(figsize=(12, 6))
    months = list(range(1, 13))
    plt.plot(months, metrics["monthly_avg"], marker="o", linewidth=2, markersize=6)
    plt.title(
        f"{crop_name.title()} - Average Monthly Price Trend ({metrics['min_year']}–{metrics['max_year']})",
        fontsize=14,
        fontweight="bold",
    )
    plt.xlabel("Month", fontsize=12)
    plt.ylabel("Price", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.xticks(months)
    plt.tight_layout()

    fig_path = crop_figs_dir / f"{crop_name}_monthly_trend.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(str(fig_path))
    plt.close()

    # 2. Average Yearly Price Trend
    plt.figure(figsize=(12, 6))
    df_sorted = metrics["df_sorted"]
    plt.plot(
        df_sorted["year"], df_sorted["avg_price"], marker="o", linewidth=2, markersize=6
    )
    plt.title(
        f"{crop_name.title()} - Average Yearly Price Trend",
        fontsize=14,
        fontweight="bold",
    )
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Average Price", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    fig_path = crop_figs_dir / f"{crop_name}_yearly_trend.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(str(fig_path))
    plt.close()

    # 3. Smoothed Trend (3-Year Moving Average)
    plt.figure(figsize=(12, 6))
    plt.plot(
        df_sorted["year"],
        df_sorted["avg_price"],
        marker="o",
        linewidth=2,
        markersize=6,
        label="Annual Average",
        solid_capstyle="round",
    )
    plt.plot(
        df_sorted["year"],
        df_sorted["moving_avg_3yr"],
        "--",
        linewidth=2,
        label="3-Year Moving Average",
    )
    plt.title(
        f"{crop_name.title()} - Yearly Trend with 3-Year Moving Average",
        fontsize=14,
        fontweight="bold",
    )
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Average Price", fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    fig_path = crop_figs_dir / f"{crop_name}_smoothed_trend.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(str(fig_path))
    plt.close()

    # 4. Year-over-Year Change (%)
    plt.figure(figsize=(12, 6))
    yoy_data = df_sorted.dropna(subset=["yoy_change_pct"])
    bars = plt.bar(yoy_data["year"], yoy_data["yoy_change_pct"], alpha=0.7)
    plt.axhline(y=0, color="black", linestyle="-", linewidth=1)
    plt.title(
        f"{crop_name.title()} - Year-over-Year Price Change (%)",
        fontsize=14,
        fontweight="bold",
    )
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("YoY Change (%)", fontsize=12)
    plt.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()

    fig_path = crop_figs_dir / f"{crop_name}_yoy_change.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(str(fig_path))
    plt.close()

    # 5. Monthly Correlation Heatmap
    plt.figure(figsize=(10, 8))
    corr_matrix = metrics["correlation_matrix"]

    # Create the heatmap using imshow
    im = plt.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)

    # Add colorbar
    cbar = plt.colorbar(im)
    cbar.set_label("Correlation Coefficient", fontsize=12)

    # Set ticks and labels
    months = [str(i) for i in range(1, 13)]
    plt.xticks(range(12), months)
    plt.yticks(range(12), months)
    plt.xlabel("Month", fontsize=12)
    plt.ylabel("Month", fontsize=12)

    # Add text annotations
    for i in range(12):
        for j in range(12):
            text = plt.text(
                j,
                i,
                f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=9,
            )

    plt.title(
        f"{crop_name.title()} - Correlation Between Monthly Prices",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()

    fig_path = crop_figs_dir / f"{crop_name}_correlation_heatmap.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    saved_figures.append(str(fig_path))
    plt.close()

    return saved_figures


def analyze_individual_crop(file_path: str) -> Dict[str, Any]:
    """
    Analyze a single crop with organized folder structure.

    Args:
        file_path: Path to the CSV file to analyze

    Returns:
        Dictionary containing analysis results
    """
    try:
        # Extract crop name from path
        crop_name = get_crop_name_from_path(file_path)

        print(f"\n🌾 Analyzing {crop_name.title()}...")
        print(f"📂 Reading data from: {file_path}")

        # Read CSV
        try:
            df = pd.read_csv(file_path)
            print(f"✓ Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
        except Exception as e:
            raise ValueError(f"Failed to read CSV file: {e}")

        # Validate and clean data
        print("🔍 Validating and cleaning data...")
        df_clean, df_long, validation_info = validate_and_clean_data(df, file_path)
        print(f"✓ Data cleaned: {len(df_long)} monthly records processed")

        if validation_info["flagged_years"]:
            print(
                f"⚠️ Warning: {len(validation_info['flagged_years'])} years flagged for sparse data"
            )

        # Calculate metrics
        print("📈 Calculating core metrics...")
        metrics = calculate_core_metrics(df_clean, df_long)
        print(f"✓ Metrics calculated for {metrics['year_range']}")

        # Create visualizations in crop-specific folder
        print("🎨 Creating visualizations...")
        figure_paths = create_crop_visualizations(metrics, crop_name)
        print(f"✓ {len(figure_paths)} charts saved to ./figs/{crop_name}/")

        # Generate summary with crop name
        print("📝 Generating summary...")
        summary_text = generate_text_summary(metrics, validation_info, crop_name)

        # Save summary to crop-specific folder
        crop_reports_dir = Path(f"./reports/{crop_name}")
        crop_reports_dir.mkdir(parents=True, exist_ok=True)
        summary_path = crop_reports_dir / f"{crop_name}_analysis_summary.md"

        with open(summary_path, "w", encoding="utf-8") as f:
            f.write(summary_text)

        print(f"✓ Summary saved to {summary_path}")

        return {
            "crop_name": crop_name,
            "figure_paths": figure_paths,
            "summary_text": summary_text,
            "metrics": metrics,
            "summary_path": str(summary_path),
            "validation_info": validation_info,
        }

    except Exception as e:
        error_msg = f"Error analyzing {file_path}: {str(e)}"
        print(f"❌ {error_msg}")
        return {
            "error": error_msg,
            "crop_name": get_crop_name_from_path(file_path),
            "figure_paths": [],
            "summary_text": "",
            "metrics": {},
            "summary_path": "",
            "validation_info": {},
        }


def run_individual_crop_analysis(file_paths: List[str]) -> None:
    """
    Run analysis on individual crops with organized folder structure.

    Args:
        file_paths: List of paths to CSV files to analyze
    """
    print("🚀 Starting individual crop analysis...")
    print(f"📋 Processing {len(file_paths)} crops with organized folder structure")

    # Extract crop names and create directories
    crop_names = [get_crop_name_from_path(path) for path in file_paths]
    create_crop_directories(crop_names)

    # Store results for master index
    master_index_data = []
    successful_analyses = []
    failed_analyses = []

    # Process each crop
    for i, file_path in enumerate(file_paths, 1):
        crop_name = get_crop_name_from_path(file_path)
        print(f"\n{'=' * 80}")
        print(f"Processing crop {i}/{len(file_paths)}: {crop_name.title()}")
        print(f"{'=' * 80}")

        result = analyze_individual_crop(file_path)

        if "error" in result:
            failed_analyses.append(
                {
                    "crop_name": result["crop_name"],
                    "file_path": file_path,
                    "error": result["error"],
                }
            )
            continue

        successful_analyses.append(result)

        # Extract data for master index
        metrics = result["metrics"]
        validation_info = result["validation_info"]

        index_entry = {
            "crop": result["crop_name"].title(),
            "year_range": metrics["year_range"],
            "overall_mean": round(metrics["overall_mean"], 2),
            "max_year": metrics["max_price_year"],
            "max_avg_price": round(metrics["max_avg_price"], 2),
            "min_year": metrics["min_price_year"],
            "min_avg_price": round(metrics["min_avg_price"], 2),
            "avg_yoy_change_pct": round(metrics["avg_yoy_change_pct"], 2),
            "std_yoy_change_pct": round(metrics["std_yoy_change_pct"], 2),
            "high_volatility_years_count": len(metrics["high_volatility_years"]),
            "missing_data_pct": round(
                validation_info["missing_summary"]["overall_missing_percentage"], 1
            ),
            "flagged_years_count": len(validation_info["flagged_years"]),
            "figures_folder": f"./figs/{result['crop_name']}/",
            "report_path": result["summary_path"],
        }

        master_index_data.append(index_entry)

    # Create and save master index
    print(f"\n{'=' * 80}")
    print("📊 INDIVIDUAL CROP ANALYSIS SUMMARY")
    print(f"{'=' * 80}")

    if master_index_data:
        master_df = pd.DataFrame(master_index_data)
        index_path = "./reports/crop_analysis_index.csv"
        master_df.to_csv(index_path, index=False)

        print(f"\n✅ Successfully processed {len(successful_analyses)} crops")
        if failed_analyses:
            print(f"❌ Failed to process {len(failed_analyses)} crops")

        print(f"\n📋 Crop Analysis Index ({index_path}):")
        print("-" * 120)

        # Print formatted table
        for _, row in master_df.iterrows():
            print(
                f"{row['crop']:<12} | {row['year_range']:<12} | Mean: {row['overall_mean']:>8.2f} | "
                f"Max: {row['max_avg_price']:>8.2f} ({row['max_year']}) | Min: {row['min_avg_price']:>8.2f} ({row['min_year']}) | "
                f"YoY: {row['avg_yoy_change_pct']:>6.1f}% | Vol: {row['std_yoy_change_pct']:>5.1f}%"
            )

        print("-" * 120)

        # Summary statistics
        print(f"\n📈 Cross-Crop Summary:")
        print(f"• Average Mean Price: {master_df['overall_mean'].mean():.2f}")
        print(
            f"• Most Volatile Crop: {master_df.loc[master_df['std_yoy_change_pct'].idxmax(), 'crop']} ({master_df['std_yoy_change_pct'].max():.1f}% std)"
        )
        print(
            f"• Least Volatile Crop: {master_df.loc[master_df['std_yoy_change_pct'].idxmin(), 'crop']} ({master_df['std_yoy_change_pct'].min():.1f}% std)"
        )
        print(
            f"• Highest Average Growth: {master_df.loc[master_df['avg_yoy_change_pct'].idxmax(), 'crop']} ({master_df['avg_yoy_change_pct'].max():.1f}% YoY)"
        )

        data_quality_issues = master_df[master_df["missing_data_pct"] > 10]
        if not data_quality_issues.empty:
            print(f"\n⚠️ Data Quality Concerns:")
            for _, row in data_quality_issues.iterrows():
                print(f"• {row['crop']}: {row['missing_data_pct']}% missing data")

        print(f"\n📁 Folder Structure Created:")
        for crop in crop_names:
            print(f"• ./figs/{crop}/ - Contains 5 visualization charts")
            print(f"• ./reports/{crop}/ - Contains detailed analysis summary")

    else:
        print("❌ No crops were successfully processed")

    # Report failed analyses
    if failed_analyses:
        print(f"\n❌ Failed Analyses:")
        for failure in failed_analyses:
            print(f"• {failure['crop_name']}: {failure['error']}")

    print(f"\n🎯 Output Summary:")
    print(f"• Individual crop charts saved to: ./figs/[crop_name]/")
    print(f"• Individual crop reports saved to: ./reports/[crop_name]/")
    print(f"• Master index: ./reports/crop_analysis_index.csv")

    print(f"\n✨ Individual crop analysis complete!")

## Execute Individual Crop Analysis

This will create separate folders for each crop containing:

- **./figs/cassava/** - 5 visualization charts for cassava
- **./figs/corn/** - 5 visualization charts for corn
- **./figs/green_bean/** - 5 visualization charts for green bean
- **./figs/soybean/** - 5 visualization charts for soybean
- **./reports/[crop_name]/** - Individual analysis summaries
- **./reports/crop_analysis_index.csv** - Master comparison index


In [139]:
# Run individual crop analysis with organized folder structure
crop_file_paths = [
    str(cassava_price_avg_year_fix),
    str(corn_price_avg_year_fix),
    str(green_bean_price_avg_year_fix),
    str(soybean_price_avg_year_fix),
]

print("🌾 Individual Crop Analysis System")
print("==================================")
print("This will create separate folders for each crop with organized outputs:")

# Show what will be created
for path in crop_file_paths:
    crop_name = get_crop_name_from_path(path)
    print(f"📁 {crop_name.title()}: ./figs/{crop_name}/ & ./reports/{crop_name}/")

print("\n🚀 Starting individual crop analysis...")

# Execute the analysis
run_individual_crop_analysis(crop_file_paths)

🌾 Individual Crop Analysis System
This will create separate folders for each crop with organized outputs:
📁 Cassava: ./figs/cassava/ & ./reports/cassava/
📁 Corn: ./figs/corn/ & ./reports/corn/
📁 Green_Bean: ./figs/green_bean/ & ./reports/green_bean/
📁 Soybean: ./figs/soybean/ & ./reports/soybean/

🚀 Starting individual crop analysis...
🚀 Starting individual crop analysis...
📋 Processing 4 crops with organized folder structure
✓ Created directories for 4 crops in ./figs/ and ./reports/

Processing crop 1/4: Cassava

🌾 Analyzing Cassava...
📂 Reading data from: c:\Users\thatt\Documents\Coding Project\Science Projects\AI Crop Land-Used\data\fix_year\cassava\price_avg.csv
✓ Data loaded: 21 rows, 13 columns
🔍 Validating and cleaning data...
✓ Data cleaned: 252 monthly records processed
📈 Calculating core metrics...
✓ Metrics calculated for 2004–2024
🎨 Creating visualizations...
✓ 5 charts saved to ./figs/cassava/
📝 Generating summary...
✓ Summary saved to reports\cassava\cassava_analysis_sum

## 🎉 Analysis Complete - Individual Crop Folders Created!

### ✅ **Successfully Created Organized Structure**

The system has created **separate folders for each of the 4 crops** with complete analysis:

#### **📁 Folder Structure:**

```
./figs/
├── cassava/          (5 charts)
├── corn/             (5 charts)
├── green_bean/       (5 charts)
└── soybean/          (5 charts)

./reports/
├── cassava/          (detailed summary)
├── corn/             (detailed summary)
├── green_bean/       (detailed summary)
├── soybean/          (detailed summary)
└── crop_analysis_index.csv (master comparison)
```

#### **📊 Each Crop Folder Contains 5 Charts:**

1. **Monthly Price Trend** - Seasonal patterns across 12 months
2. **Yearly Price Trend** - Long-term price evolution
3. **Smoothed Trend** - 3-year moving average analysis
4. **Year-over-Year Changes** - Annual volatility patterns
5. **Correlation Heatmap** - Monthly price relationships

#### **📈 Key Findings Summary:**

- **Most Volatile:** Cassava (30.4% std deviation)
- **Highest Growth:** Cassava (+8.6% avg YoY)
- **Highest Prices:** Green Bean (avg 24.34)
- **Most Stable:** Soybean (14.1% std deviation)

All crops show **complete data coverage (0% missing)** from 2004-2024 with detailed trend analysis, volatility assessment, and seasonality insights available in individual reports.
